# 07 · Exportar modelo estrella para Power BI
Genera `data/processed/powerbi/*.csv` listos para importar.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
out = config.DATA_PROCESSED / 'powerbi'; out.mkdir(exist_ok=True)
df = pd.read_parquet(config.DATA_INTERIM / 'visitas_features.parquet')
ev = pd.read_parquet(config.DATA_INTERIM / 'event_log.parquet')

In [ ]:
fact_visita = df[['visit_occurrence_id','person_id','visit_start_datetime','visit_end_datetime','tipo_visita','sexo','edad',
                  'hora_llegada','franja','los_h','estancia_prolongada','ocupacion_llegada','n_etapas',
                  'los_pred_h','prob_prolongada']].copy()
fact_visita['fecha'] = fact_visita['visit_start_datetime'].dt.date
fact_visita.to_csv(out / 'fact_visita.csv', index=False)

ev[['visit_occurrence_id','orden','etapa','etapa_anterior','visit_detail_start_datetime','visit_detail_end_datetime',
    'duracion_h','espera_previa_h']].to_csv(out / 'fact_etapa.csv', index=False)

fechas = pd.date_range(df['visit_start_datetime'].min().normalize(), df['visit_start_datetime'].max().normalize())
dim_fecha = pd.DataFrame({'fecha': fechas.date, 'anio': fechas.year, 'mes': fechas.month,
                          'nombre_mes': fechas.month_name(), 'dia_semana': fechas.dayofweek,
                          'nombre_dia': fechas.day_name(), 'fin_de_semana': (fechas.dayofweek>=5).astype(int)})
dim_fecha.to_csv(out / 'dim_fecha.csv', index=False)

import shutil
for f in ['transiciones.csv','duracion_etapas.csv','ocupacion_horaria.csv','importancia_shap.csv',
          'pronostico_ocupacion.csv','alerta_saturacion.csv','simulacion_escenarios.csv']:
    if (config.DATA_PROCESSED / f).exists():
        shutil.copy(config.DATA_PROCESSED / f, out / f)
sorted(p.name for p in out.iterdir())